In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

gt_file = "/home/thanet/Work/SemEval-2026-Task5-Narrative-Understanding/semeval26-05-scripts/data/test_labels.json"
pred_file = "/home/thanet/Work/SemEval-2026-Task5-Narrative-Understanding/semeval26-05-scripts/predictions/results_test_gpt-5-mini_improved_criteria_None_20260212_163429.json_predictions.jsonl"


In [ ]:
with open(gt_file, "r") as f:
    gt_data = json.load(f)

pred_dict = {}
with open(pred_file, "r") as f:
    for line in f:
        data = json.loads(line)
        pred_dict[data["id"]] = data["prediction"]

common_ids = sorted(set(gt_data.keys()).intersection(pred_dict.keys()))

all_gt_scores = []
all_predictions = []

gold_means = []
mae_list = []

for id_ in common_ids:
    sample = gt_data[id_]
    pred = pred_dict[id_]

    # All raw GT ratings
    all_gt_scores.append(sample["average"])
    all_predictions.append(pred)

    # Mean + MAE
    mean_gt = sample["average"]
    mae = abs(pred - mean_gt)

    gold_means.append(mean_gt)
    mae_list.append(mae)

print(len(all_predictions), len(all_gt_scores))

gold_means = np.array(gold_means)
mae_list = np.array(mae_list)


# Overlay Histogram - score dist.
plt.figure()
bins = np.arange(1, 7) - 0.5
plt.hist(all_gt_scores, bins=bins, alpha=0.6, label="Gold Average")
plt.hist(all_predictions, bins=bins, alpha=0.6, label="Prediction")
plt.title("Distribution of Ground Truth vs Predicted Scores")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.legend()
plt.show()

# MAE by Gold Rating Range bar chart
bins = [1.0, 2.5, 3.5, 4.5, 5.0]
bin_labels = ["[1.0–2.5)", "[2.5–3.5)", "[3.5–4.5)", "[4.5–5.0]"]

bin_mae = []
bin_counts = []

for i in range(len(bins) - 1):
    if i < len(bins) - 2:
        mask = (gold_means >= bins[i]) & (gold_means < bins[i+1])
    else:
        mask = (gold_means >= bins[i]) & (gold_means <= bins[i+1])

    if np.sum(mask) > 0:
        bin_mae.append(np.mean(mae_list[mask]))
        bin_counts.append(np.sum(mask))
    else:
        bin_mae.append(0)
        bin_counts.append(0)

plt.figure()
plt.bar(bin_labels, bin_mae)

plt.title("MAE by Gold Rating Range")
plt.xlabel("Gold Rating Range")
plt.ylabel("Mean Absolute Error")
plt.ylim(0.5,0.9)
plt.show()

for label, mae, count in zip(bin_labels, bin_mae, bin_counts):
    print(f"{label}: MAE = {mae:.3f}, n = {count}")